In [1]:
%reset -f

In [2]:
debug = False

In [3]:
import os,sys
current_path = os.getcwd()

In [4]:
name = current_path.split('\\')[-1].split('_')
data = name[0]

In [5]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("D:/work/scorecard") 
base_path = 'D:/work/06.std_product/wxjk_test/'
 
FILE_PATH = base_path + f'{data}/file/'
DATA_PATH = base_path + f'{data}/data/'
TMP_PATH  = base_path + f'{data}/tmp/'
DATA_OR_PATH = base_path + f'data_or/'

import os
import gc
import pandas as pd 
import numpy as np  
import math  
import zipfile
import matplotlib.pyplot as plt
import copy
import seaborn as sns
from pylab import mpl
from ScoreCard.creat_report import Report
mpl.rcParams['font.sans-serif'] = ['SimHei']
mpl.rcParams['axes.unicode_minus'] = False  

isExists=os.path.exists(FILE_PATH)
if not isExists:
    os.makedirs(FILE_PATH) 
    
isExists=os.path.exists(DATA_PATH)
if not isExists:
    os.makedirs(DATA_PATH)

isExists=os.path.exists(TMP_PATH)
if not isExists:
    os.makedirs(TMP_PATH)  

import warnings
warnings.filterwarnings("ignore")

## 1. feature 结构查看

In [6]:
if data == 'delta':
    dataV = data + 'V1'
else:
    dataV = data + 'V2'

In [7]:
the_data_path = DATA_OR_PATH + f'维信20251106_d1_t2_z2_{dataV}_result.zip'
parquet_zip_path= the_data_path.replace(".zip", ".parquet")

In [ ]:
with zipfile.ZipFile(the_data_path, 'r') as zip_file:
    file_list = zip_file.namelist()
    print("压缩包内文件:", file_list)
    csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
    with zip_file.open(csv_filename) as csv_file:
        data_tmp = pd.read_csv(csv_file, nrows=100)

压缩包内文件: ['╬¼╨┼20251106_d1_t2_z2_deltaV1_result.txt']


In [9]:
data_tmp

,name,mobile,idCard,reqToken,backPointTime,TZ_0000_m1,TZ_0000_m12,TZ_0000_m15,TZ_0000_m18,TZ_0000_m2,...,TZ_2024_T4_m4,TZ_2024_T4_m5,TZ_2024_T4_m6,TZ_2024_T4_m7,TZ_2024_T4_m8,TZ_2024_T4_m9,TZ_2025_T1_m24,TZ_2025_T2_m24,TZ_2025_T3_m24,TZ_2025_T4_m24
0,729599939935314ad7796963a4ac81f4,6afc7f594e81633d58bf091562d0ae43,8b3a71e6517de7391e5f2fc1f20eb778,1,2025/1/22,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,d74b62337c90fb492cf1d27a35500e79,02947e5fb54d2203bf03f2a13f13d92e,58d6eedcf45eebfed22b506a03e8feb7,2,2025/9/20,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0deee47fffaa56cab2b085c3b173d2eb,642f657575362a912b6ce80606b79cf9,d3f71f697b5f0a8cbde4772e7bb18f06,3,2024/11/29,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,8dd18d524f30529d8f0910dbd2d015af,71d234b2a1b0ee10fb36229d8575f6c9,8f5161712dc8c618598ad4ec0632036a,4,2024/12/26,0,3,3,3,0,...,0,0,0,0,0,0,0,0,0,0
4,2ecc12f6df07e5d0b982592df0c0924f,eb2d382e9f02be6e4d24498d324a64d1,98cfd4e53010fcede77dcde9bd7787be,5,2024/10/22,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,f712d041d97638c1d95fe8fbcf9b3fda,680bab1a2eded00535fb9b3425fe6f6a,0f7ca5a6505d58de44b97e718782984e,96,2024/11/23,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
96,f7364b06e9eba56c09250b6c4a8a5956,4197b7e68c21092d731f501572e88027,2ab88ae7397ed1bed5c244a7a815db57,97,2024/12/7,0,9,9,9,0,...,0,0,0,0,0,0,0,0,0,0
97,0558e2cf3edbcf1dc3f57867ca381024,11b38b7ba131789680d336e5f28b95c6,9bd6cadc189e9e81d0dcffaff9555041,98,2024/12/23,0,5,5,5,1,...,0,0,0,0,0,0,0,0,0,0
98,fae976114952812aec007041aed866e8,640cd560eb0fd51bdf56eb3481bd1fff,a93ad3f9bb204037870d2aff28b0c53b,99,2024/11/29,0,2,2,2,1,...,0,0,0,0,0,0,0,0,0,0


In [10]:
data_tmp.select_dtypes('O')

,name,mobile,idCard,backPointTime
0,729599939935314ad7796963a4ac81f4,6afc7f594e81633d58bf091562d0ae43,8b3a71e6517de7391e5f2fc1f20eb778,2025/1/22
1,d74b62337c90fb492cf1d27a35500e79,02947e5fb54d2203bf03f2a13f13d92e,58d6eedcf45eebfed22b506a03e8feb7,2025/9/20
2,0deee47fffaa56cab2b085c3b173d2eb,642f657575362a912b6ce80606b79cf9,d3f71f697b5f0a8cbde4772e7bb18f06,2024/11/29
3,8dd18d524f30529d8f0910dbd2d015af,71d234b2a1b0ee10fb36229d8575f6c9,8f5161712dc8c618598ad4ec0632036a,2024/12/26
4,2ecc12f6df07e5d0b982592df0c0924f,eb2d382e9f02be6e4d24498d324a64d1,98cfd4e53010fcede77dcde9bd7787be,2024/10/22
...,...,...,...,...
95,f712d041d97638c1d95fe8fbcf9b3fda,680bab1a2eded00535fb9b3425fe6f6a,0f7ca5a6505d58de44b97e718782984e,2024/11/23
96,f7364b06e9eba56c09250b6c4a8a5956,4197b7e68c21092d731f501572e88027,2ab88ae7397ed1bed5c244a7a815db57,2024/12/7
97,0558e2cf3edbcf1dc3f57867ca381024,11b38b7ba131789680d336e5f28b95c6,9bd6cadc189e9e81d0dcffaff9555041,2024/12/23
98,fae976114952812aec007041aed866e8,640cd560eb0fd51bdf56eb3481bd1fff,a93ad3f9bb204037870d2aff28b0c53b,2024/11/29


In [11]:
object_columns = []
for column in data_tmp.select_dtypes('O').columns:
    if column not in ['name','idCard','mobile','reqToken','backPointTime','version']:
        print(column)
        object_columns.append(column)

In [12]:
fea_list =list(data_tmp.columns[5:])

In [13]:
fea_list

['TZ_0000_m1',
 'TZ_0000_m12',
 'TZ_0000_m15',
 'TZ_0000_m18',
 'TZ_0000_m2',
 'TZ_0000_m24',
 'TZ_0000_m3',
 'TZ_0000_m4',
 'TZ_0000_m5',
 'TZ_0000_m6',
 'TZ_0000_m9',
 'TZ_0000_w1',
 'TZ_0000_w2',
 'TZ_0000_w3',
 'TZ_0000_w4',
 'TZ_0001_m12',
 'TZ_0001_m15',
 'TZ_0001_m18',
 'TZ_0001_m2',
 'TZ_0001_m24',
 'TZ_0001_m3',
 'TZ_0001_m4',
 'TZ_0001_m5',
 'TZ_0001_m6',
 'TZ_0001_m9',
 'TZ_0001_w2',
 'TZ_0001_w3',
 'TZ_0001_w4',
 'TZ_0002_m12',
 'TZ_0002_m15',
 'TZ_0002_m18',
 'TZ_0002_m2',
 'TZ_0002_m24',
 'TZ_0002_m3',
 'TZ_0002_m4',
 'TZ_0002_m5',
 'TZ_0002_m6',
 'TZ_0002_m9',
 'TZ_0002_w2',
 'TZ_0002_w3',
 'TZ_0002_w4',
 'TZ_0003_m12',
 'TZ_0003_m15',
 'TZ_0003_m18',
 'TZ_0003_m2',
 'TZ_0003_m24',
 'TZ_0003_m3',
 'TZ_0003_m4',
 'TZ_0003_m5',
 'TZ_0003_m6',
 'TZ_0003_m9',
 'TZ_0003_w2',
 'TZ_0003_w3',
 'TZ_0003_w4',
 'TZ_0004_m1',
 'TZ_0004_m12',
 'TZ_0004_m15',
 'TZ_0004_m18',
 'TZ_0004_m2',
 'TZ_0004_m24',
 'TZ_0004_m3',
 'TZ_0004_m4',
 'TZ_0004_m5',
 'TZ_0004_m6',
 'TZ_0004_m9',
 'TZ_

In [14]:
len(fea_list)

15160

## 2 转parquet

In [24]:
from new_tools.parquet_utils import prepare_parquet_from_csv_new

In [ ]:
zip_path = the_data_path
force_regenerate = False
encoding = 'utf8'
separator = ','
enable_schema = True
feature_start_index = 5
row_group_size = 10000
batch_size = 10000         
mem_threshold_gb = 16.0  

In [25]:
prepare_parquet_from_csv_new(
    zip_path, parquet_zip_path, force_regenerate, encoding, separator, 
    enable_schema, feature_start_index, row_group_size,
    batch_size=batch_size, 
    mem_threshold_gb=mem_threshold_gb
)

开始转换任务 '维信20251106_d1_t2_z2_deltaV1_result.zip' -> '维信20251106_d1_t2_z2_deltaV1_result.parquet'
  - 参数: batch_size=10000, row_group_size=10000, mem_threshold=16.0GB
  - 检测到ZIP文件，准备解压至临时目录: C:\Users\30469\AppData\Local\Temp\tmp664vf930
  - 正在解压: ╬¼╨┼20251106_d1_t2_z2_deltaV1_result.txt
  - 使用高效的 UTF-8 流式路径...
  - 阶段1: 准备混合类型Schema...
✅ 成功构建混合类型Schema，共 15165 列。
  - 阶段2: 开始流式读取并分块写入Parquet文件...
    - 初始化 ParquetWriter, row_group_size: 10000
    - 已处理 5 批次, 累计 50,235 行, 内存: 5.39GB
    - 已处理 10 批次, 累计 100,449 行, 内存: 3.98GB
    - 已处理 15 批次, 累计 150,675 行, 内存: 6.38GB
    - 已处理 20 批次, 累计 200,761 行, 内存: 8.36GB
    - 已处理 25 批次, 累计 250,767 行, 内存: 10.52GB
    - 已处理 30 批次, 累计 300,834 行, 内存: 13.15GB
    - 已处理 35 批次, 累计 350,902 行, 内存: 2.39GB
    - 已处理 40 批次, 累计 401,021 行, 内存: 4.90GB
    - 已处理 45 批次, 累计 451,121 行, 内存: 7.24GB
    - 已处理 50 批次, 累计 501,212 行, 内存: 9.56GB
    - 已处理 55 批次, 累计 551,304 行, 内存: 11.85GB
    - 已处理 60 批次, 累计 601,388 行, 内存: 14.15GB
    - 已处理 65 批次, 累计 651,777 行, 内存: 2.54GB
    - 已处理

## 3 feature selection 1

In [27]:
print('debug mode: ', debug)

debug mode:  False


In [28]:
if debug:
    fea_list = fea_list[:1000]

In [29]:
from new_tools import flitter_by_std_fpr
ftr1_keep_dict = {}
length = 500
for i in range(0,(len(fea_list) // length) + 1):
    start,end  = i * length, min((i + 1) * length, len(fea_list))
    print(start, ' - ', end)
    tmp_lst = [i for i in fea_list[start:end] if i not in object_columns]
    each_df = pd.read_parquet(parquet_zip_path, 
                    columns=tmp_lst,
                    engine='pyarrow') 
    ftr1_keep = flitter_by_std_fpr.drop_svr(each_df.fillna(-999), 0.95, tmp_lst)
    ftr1_keep = flitter_by_std_fpr.drop_std(each_df.fillna(-999), 0, ftr1_keep)
    ftr1_keep_dict[i] = ftr1_keep
    del each_df, ftr1_keep
    gc.collect()
    

0  -  500


筛选单一值率: 100%|██████████| 500/500 [00:02<00:00, 214.51it/s]


舍弃了 242 个单一值占比超过 95.0% 的列


筛选标准差: 100%|██████████| 258/258 [00:01<00:00, 225.34it/s]


舍弃了 0 个标准差小于等于 0 的列
500  -  1000


...

舍弃了 0 个标准差小于等于 0 的列
15000  -  15160


筛选单一值率: 100%|██████████| 160/160 [00:00<00:00, 302.45it/s]


舍弃了 148 个单一值占比超过 95.0% 的列


筛选标准差: 100%|██████████| 12/12 [00:00<00:00, 220.46it/s]

舍弃了 0 个标准差小于等于 0 的列


In [30]:
fea_list = []
for k,v in ftr1_keep_dict.items():
    fea_list.extend(v)

In [31]:
fea_list = list(set(fea_list))
fea_list

['TZ_1247_m18',
 'TZ_1812_m18',
 'TZ_0083_w4',
 'TZ_1640_m1',
 'TZ_1747_m24',
 'TZ_0272_m2',
 'TZ_0160_m5',
 'TZ_0005_m15',
 'TZ_0390_m8',
 'TZ_1220_m12',
 'TZ_1870_m24',
 'TZ_1752_m24',
 'TZ_0442_m4',
 'TZ_1711_m12',
 'TZ_0011_m2',
 'TZ_0143_m12',
 'TZ_2019_T3_m24',
 'TZ_0017_m24',
 'TZ_0063_m4',
 'TZ_1225_m3',
 'TZ_1512_m12',
 'TZ_0028_m18',
 'TZ_0338_m9',
 'TZ_0415_m5',
 'TZ_1731_m18',
 'TZ_1972_m18',
 'TZ_1702_m18',
 'TZ_0036_w4',
 'TZ_0033_m12',
 'TZ_0333_m6',
 'TZ_1896_m24',
 'TZ_1103',
 'TZ_0077_m6',
 'TZ_0087_m24',
 'TZ_1358_m6',
 'TZ_0032_w4',
 'TZ_1424_m24',
 'TZ_0141_m12',
 'TZ_0415_m12',
 'TZ_0273_m3',
 'TZ_1521_m1',
 'TZ_1420_m1',
 'TZ_0055_m24',
 'TZ_1144_m12',
 'TZ_0328_m4',
 'TZ_1318_m12',
 'TZ_0088_m9',
 'TZ_1643_m12',
 'TZ_0183_m2',
 'TZ_1772_m6',
 'TZ_1782_m24',
 'TZ_1904_m3',
 'TZ_0108_m18',
 'TZ_1347_m12',
 'TZ_1490_m24',
 'TZ_1544_m12',
 'TZ_0463_m12',
 'TZ_1222_m12',
 'TZ_1546_m12',
 'TZ_1620_m6',
 'TZ_0020_m12',
 'TZ_1672_m6',
 'TZ_0396_m10',
 'TZ_1875_m18',
 'T

In [32]:
pd.DataFrame({'var_names': fea_list}).to_csv(DATA_PATH + 'fea_list.csv')

In [15]:
fea_list = pd.read_csv(DATA_PATH + 'fea_list.csv')
fea_list = fea_list['var_names'].tolist()
fea_list

['TZ_1247_m18',
 'TZ_1812_m18',
 'TZ_0083_w4',
 'TZ_1640_m1',
 'TZ_1747_m24',
 'TZ_0272_m2',
 'TZ_0160_m5',
 'TZ_0005_m15',
 'TZ_0390_m8',
 'TZ_1220_m12',
 'TZ_1870_m24',
 'TZ_1752_m24',
 'TZ_0442_m4',
 'TZ_1711_m12',
 'TZ_0011_m2',
 'TZ_0143_m12',
 'TZ_2019_T3_m24',
 'TZ_0017_m24',
 'TZ_0063_m4',
 'TZ_1225_m3',
 'TZ_1512_m12',
 'TZ_0028_m18',
 'TZ_0338_m9',
 'TZ_0415_m5',
 'TZ_1731_m18',
 'TZ_1972_m18',
 'TZ_1702_m18',
 'TZ_0036_w4',
 'TZ_0033_m12',
 'TZ_0333_m6',
 'TZ_1896_m24',
 'TZ_1103',
 'TZ_0077_m6',
 'TZ_0087_m24',
 'TZ_1358_m6',
 'TZ_0032_w4',
 'TZ_1424_m24',
 'TZ_0141_m12',
 'TZ_0415_m12',
 'TZ_0273_m3',
 'TZ_1521_m1',
 'TZ_1420_m1',
 'TZ_0055_m24',
 'TZ_1144_m12',
 'TZ_0328_m4',
 'TZ_1318_m12',
 'TZ_0088_m9',
 'TZ_1643_m12',
 'TZ_0183_m2',
 'TZ_1772_m6',
 'TZ_1782_m24',
 'TZ_1904_m3',
 'TZ_0108_m18',
 'TZ_1347_m12',
 'TZ_1490_m24',
 'TZ_1544_m12',
 'TZ_0463_m12',
 'TZ_1222_m12',
 'TZ_1546_m12',
 'TZ_1620_m6',
 'TZ_0020_m12',
 'TZ_1672_m6',
 'TZ_0396_m10',
 'TZ_1875_m18',
 'T

In [16]:
len(fea_list)

6220

## 4 label process

In [17]:
import gc
gc.collect()

0

In [18]:
label = pd.read_csv(DATA_OR_PATH + 'wxjk_data.csv')

In [19]:
label.head()

,id_no_md5,id_no_sha256,mobile_md5,mobile_sha256,name_md5,name_sha256,auth_time,devflag,t10,mob2m2,mob4m2,mob7m2,ever_m2,train_or_test,weight,if_app,cust_type
0,8b3a71e6517de7391e5f2fc1f20eb778,0cda4b6dc3689892b6dd352682ab992f1133e352da5b67...,6afc7f594e81633d58bf091562d0ae43,825cff373793a29d4b54a5b79ffd675e7bfa3f9994e17a...,729599939935314ad7796963a4ac81f4,285914ed0f7dfffcd51e4932bf3c0efea0d30ba037947d...,2025-01-22 13:31:51,新贷,0,0,0,0,0,train,2.399906,NaN,0
1,58d6eedcf45eebfed22b506a03e8feb7,d230035d3290ec890f65b7e4799ae0fcc18a5c4736823b...,02947e5fb54d2203bf03f2a13f13d92e,6f4cafd6770dcc2541bf465b1bf96484a97f919732c8ed...,d74b62337c90fb492cf1d27a35500e79,35949913a1731771be314e4f962ee3d29f6125a5c1d42b...,2025-09-20 11:24:34,缓存测试,-1,-1,-1,-1,-1,NaN,1.000000,NaN,0
2,d3f71f697b5f0a8cbde4772e7bb18f06,b6b019e7109962bf72047627c996e1f88c0ccda5fa52fe...,642f657575362a912b6ce80606b79cf9,54e5b2dae4064667605de2629fc16aa3bfefcd8ddc59ed...,0deee47fffaa56cab2b085c3b173d2eb,a63f7347fcae08bf743cf765655ba9dd3c561b5a729ef5...,2024-11-29 21:15:36,新贷,0,0,0,0,0,test,2.399906,NaN,0
3,8f5161712dc8c618598ad4ec0632036a,07d75636e2960ed5ceed4fd27c5de4d127dd2f73d48a0b...,71d234b2a1b0ee10fb36229d8575f6c9,8c2099a4ffb5950eb5fe1cad3dcfc4f9cfba1b9a7ef778...,8dd18d524f30529d8f0910dbd2d015af,676bc534e5cef03794ac310b12d4f0e77f1dfb92c834a3...,2024-12-26 07:31:29,新贷,0,0,0,0,0,test,2.399906,NaN,0
4,98cfd4e53010fcede77dcde9bd7787be,05590f96fbf327fe2dfda4ffb55274ee39f75d9d5e2165...,eb2d382e9f02be6e4d24498d324a64d1,831c5d53539eeadc6f06acf271b59318efe5c8bfccb1bf...,2ecc12f6df07e5d0b982592df0c0924f,60497f66353f01dd2bd1eee45b381958c0912534171f76...,2024-10-22 09:27:24,新贷,0,0,0,0,0,test,2.399906,NaN,0


In [20]:
label = label[['mobile_md5','auth_time','mob4m2','devflag','weight','if_app']]
label.rename(columns={'mobile_md5':'mobile','auth_time':'backPointTime','mob4m2':'label'},inplace=True)
label['backPointTime'] = pd.to_datetime(label['backPointTime']).dt.strftime('%Y-%m-%d')


In [21]:
label['new_flag'] = label.apply(lambda x: x['devflag']+'_'+str(int(x['if_app'])) if x['if_app'] == 1 or x['if_app'] == 0 else x['devflag'], axis=1)

In [22]:
label['new_flag'].value_counts()

新贷         244144
复贷_1       226015
复贷_0       103905
新贷基测        73029
复贷基测        69127
评分进件-复贷      3200
评分进件-新贷      3200
缓存测试          918
回溯测试          500
Name: new_flag, dtype: int64

In [23]:
label = label[label['new_flag'].isin(['新贷', '复贷_1', '复贷_0','新贷基测','复贷基测'])]

In [24]:
label.drop(['if_app'], axis=1, inplace=True)

In [25]:
label.head()

,mobile,backPointTime,label,devflag,weight,new_flag
0,6afc7f594e81633d58bf091562d0ae43,2025-01-22,0,新贷,2.399906,新贷
2,642f657575362a912b6ce80606b79cf9,2024-11-29,0,新贷,2.399906,新贷
3,71d234b2a1b0ee10fb36229d8575f6c9,2024-12-26,0,新贷,2.399906,新贷
4,eb2d382e9f02be6e4d24498d324a64d1,2024-10-22,0,新贷,2.399906,新贷
5,e671a7d592ac58d7f44710b3902f2acd,2024-10-17,0,新贷,1.000000,新贷


## 5 label merge

In [31]:
from new_tools.reduce_mem_usage import reduce_mem_usage_dic 

def chunk_reduce_mem_usage(file_name,columns):
    chunks = pd.read_csv(file_name, usecols=columns, chunksize=200_000)
    chunk_list = []
    for chunk in chunks:
        chunk_reduce_mem = reduce_mem_usage_dic(chunk)
        del chunk
        chunk_list.append(chunk_reduce_mem)
        del chunk_reduce_mem
    df = pd.concat(chunk_list)
    return df

In [32]:
object_columns

[]

In [33]:
gc.collect()

0

In [ ]:
import polars as pl
from new_tools.reduce_mem_usage import reduce_mem_usage_dic

processed_chunks = []
label_index_lst = (label['mobile'] + label['backPointTime']).to_list()
label_tmp = label.copy()

use_cols = ['mobile', 'backPointTime'] + fea_list + object_columns

# 惰性扫描，不实际加载数据（不指定列）
lazy_df = pl.scan_parquet(parquet_zip_path).select(use_cols)

# 获取总行数（这个会执行一次快速扫描）
total_rows = lazy_df.select(pl.count()).collect().item()
print(f"总行数: {total_rows}")

batch_size = 100_000
i = 1

# 使用 slice 分批获取数据
for offset in range(0, total_rows, batch_size):
    # 惰性切片 + collect 只加载当前批次
    chunk = lazy_df.slice(offset, batch_size).collect().to_pandas()
    
    if len(chunk) == 0:
        continue
        
    temp_df = reduce_mem_usage_dic(chunk)
    del chunk
    gc.collect()
    
    temp_df.drop_duplicates(['mobile', 'backPointTime'], inplace=True)
    print(f"Chunk {i}: 去重后 {len(temp_df)} 行")
    
    temp_df['backPointTime'] = pd.to_datetime(temp_df['backPointTime']).dt.strftime('%Y-%m-%d')            
    temp_df = pd.merge(label_tmp, temp_df, how='inner', on=['mobile', 'backPointTime'])
    
    remove_set = set((temp_df['mobile'] + temp_df['backPointTime']).to_list())
    label_index_lst = list(set(label_index_lst) - remove_set)
    label_tmp = label[(label['mobile'] + label['backPointTime'].astype(str)).isin(label_index_lst)]
    
    temp_df.to_parquet(DATA_PATH + f'chunk_{i}.parquet')
    processed_chunks.append(temp_df)
    print(f"Chunk {i}: 保存完成, 剩余未匹配: {len(label_index_lst)}")
    
    del temp_df, remove_set
    gc.collect()
    i += 1

# 处理 mobile 为空的记录
mobile_num_df = label[pd.isna(label['mobile'])]
mobile_num_df[fea_list] = np.nan
processed_chunks.append(mobile_num_df)

print(f"最终未匹配数量: {len(label_index_lst)}")

总行数: 724038
当前内存占用: 2374.27 MB


100%|██████████| 6222/6222 [00:03<00:00, 1595.08it/s]


最终内存占用: 2374.27 MB
下降了 0.0%
Chunk 1: 去重后 99990 行
Chunk 1: 保存完成, 剩余未匹配: 617195
当前内存占用: 2374.27 MB


100%|██████████| 6222/6222 [00:05<00:00, 1172.38it/s]


最终内存占用: 2374.27 MB
下降了 0.0%
Chunk 2: 去重后 99981 行
Chunk 2: 保存完成, 剩余未匹配: 518222
当前内存占用: 2374.27 MB


...

最终内存占用: 2374.27 MB
下降了 0.0%
Chunk 6: 去重后 99990 行
Chunk 6: 保存完成, 剩余未匹配: 122642
当前内存占用: 2374.27 MB


100%|██████████| 6222/6222 [00:04<00:00, 1453.05it/s]


最终内存占用: 2374.27 MB
下降了 0.0%
Chunk 7: 去重后 99989 行
Chunk 7: 保存完成, 剩余未匹配: 23759
当前内存占用: 570.73 MB


100%|██████████| 6222/6222 [00:02<00:00, 2355.81it/s]


最终内存占用: 570.73 MB
下降了 0.0%
Chunk 8: 去重后 24037 行
Chunk 8: 保存完成, 剩余未匹配: 0
最终未匹配数量: 0


In [36]:
# from new_tools.reduce_mem_usage import reduce_mem_usage_dic

# processed_chunks = []
# label_index_lst = (label['mobile'] + label['backPointTime']).to_list()
# label_tmp = label.copy()

# with zipfile.ZipFile(DATA_OR_PATH + '信飞20251023_d1_t2_z2_deltaV1_result.zip', 'r') as zip_file:
#     file_list = zip_file.namelist()
#     csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
    
#     with zip_file.open(csv_filename) as csv_file:
#         chunks = pd.read_csv(csv_file, usecols=['mobile','backPointTime'] + fea_list + object_columns, chunksize=200_000)
#         i = 1
#         for chunk in chunks:  # 在 with 块内完成迭代
#             temp_df = reduce_mem_usage_dic(chunk)
#             temp_df.drop_duplicates(['mobile', 'backPointTime'], inplace=True)
#             print(1)
#             temp_df['backPointTime'] = pd.to_datetime(temp_df['backPointTime']).dt.strftime('%Y-%m-%d')            
#             print(2)
#             temp_df = pd.merge(label_tmp, temp_df, how='inner', on=['mobile', 'backPointTime'])
#             print(3)
#             remove_set = set((temp_df['mobile'] + temp_df['backPointTime']).to_list())
#             label_index_lst = list(set(label_index_lst) - remove_set)
#             label_tmp = label[(label['mobile'] + label['backPointTime'].astype(str)).isin(label_index_lst)]
#             print(4)
#             # processed_chunks.append(temp_df)
#             temp_df.to_parquet(DATA_PATH + f'chunk_{i}.parquet')
#             print(5)
#             del chunk, temp_df, remove_set
#             gc.collect()
#             i = i+1
#         mobile_num_df = label[pd.isna(label['mobile'])]  # 或者根据实际逻辑调整
#         mobile_num_df[fea_list] = np.nan
#         # processed_chunks.append(mobile_num_df)

# print(len(label_index_lst))


In [ ]:
label_index_lst

[]

In [27]:
gc.collect()

0

In [26]:
from new_tools.parquet_utils import concat_parquet_streaming

files = [DATA_PATH + f"chunk_{i}.parquet" for i in range(1,9)]
# print(files)
concat_parquet_streaming(
    files,
    output_path=DATA_OR_PATH + f"{data}_data_all1.parquet",
    batch_size=50_000,     # 可按内存调小
    mem_threshold_gb=8.0,  # 建议设为物理内存的30%~50%
    columns_consistent=True  # 如果所有文件的列都一致，设为True以提速
)

正在处理文件 1/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_1.parquet
正在处理文件 2/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_2.parquet
正在处理文件 3/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_3.parquet
正在处理文件 4/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_4.parquet
正在处理文件 5/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_5.parquet
正在处理文件 6/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_6.parquet
正在处理文件 7/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_7.parquet
正在处理文件 8/8: D:/work/06.std_product/wxjk_test/delta/data/chunk_8.parquet


In [30]:
data_all1 = pd.read_parquet(DATA_OR_PATH + f"{data}_data_all1.parquet")

: 

: 

In [ ]:
data_all1 = pd.concat(processed_chunks, ignore_index=True)
del processed_chunks
gc.collect()

In [ ]:
data_all1.shape

NameError: name 'data_all1' is not defined

In [25]:
data_all1.head()

,ms_no,mobile_sha256,hs_date,prod_flag,sample_type,target1_mi,target2_mi,stage,TZ_0000_m1,TZ_0000_m12,...,TZ_0073_m2,TZ_0073_m24,TZ_0073_m3,TZ_0073_m4,TZ_0073_m5,TZ_0073_m6,TZ_0073_m9,TZ_0073_w2,TZ_0073_w3,TZ_0073_w4
0,2618468494244577696,d45735e2015b20c4e8f95645096ae0dac3c9924b9e6b35...,2024-11-21,S,CA,0.0,0.0,test,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
1,2626479791808743072,dff94df987f14dd7304d25f291729f0b4926adad0df39b...,2024-12-02,S,CA,0.0,0.0,test,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2582720644667933856,4d4d2304e9265b029491e1fec60e7431d55285fcad426e...,2024-10-03,S,CA,0.0,0.0,train,0.0,8.0,...,0.0,3.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0
3,2597987997622206880,f05c3b1367d78a5da0b060110d8724e72664c86dc456a4...,2024-10-24,S,CA,0.0,0.0,train,0.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,0.0,0.0,1.0
4,2654564558907114656,d92c8174bf3dd98aa5d11edefef13f24512c53b694071d...,2025-01-10,S,CA,0.0,NaN,test,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
data_all1['ms_no'] = data_all1['ms_no'].astype('str')

In [27]:
data_all1.to_parquet(DATA_PATH + 'data_all1.parquet')

In [28]:
# data_all1 = pd.read_parquet(DATA_PATH + 'data_all1.parquet')

In [29]:
# data_all1[object_columns] = data_all1[object_columns].apply(
#     lambda x: x.astype('category').cat.codes.replace({-1: np.nan}).astype('float64')
# )

In [30]:
data_all1[object_columns] 

""
0
1
2
3
4
...
2000573
2000574
2000575
2000576


In [31]:
# data_all1[object_columns].count()

In [32]:
data_all1[data_all1['sample_type'] == 'CA'].to_parquet(DATA_PATH + 'data_all1_CA.parquet')

In [33]:
gc.collect()

0

In [34]:
data_all1[data_all1['sample_type'] == 'CR'].to_parquet(DATA_PATH + 'data_all1_CR.parquet')

In [35]:
%reset -f